# Notes to self:
Currently, I am able to get the model to sample with batch dimensions and I am able to use the following internal methods:

* `sample_unconditional_prior`
* `sample_conditional_prior`
* `sample_unconditional_posterior`
* `sample_conditional_posterior`

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import pytensor.tensor as pt

from pymc_extras.statespace.core.statespace import PyMCStateSpace
from pymc_extras.statespace.filters import StandardFilter, KalmanSmoother

from pymc_extras.statespace.core.properties import (
    Parameter,
    State,
    Shock,
    Coord,
)
from pymc_extras.statespace.utils.constants import ALL_STATE_DIM, ALL_STATE_AUX_DIM, SHOCK_DIM

In [ ]:
class AutoRegressiveThree(PyMCStateSpace):
    def __init__(self, mode: str):
        k_states = 3  # size of the state vector x
        k_posdef = 1  # number of shocks (size of the state covariance matrix Q)
        k_endog = 1  # number of observed states
        batch_size = 2

        super().__init__(
            k_endog=k_endog, k_states=k_states, k_posdef=k_posdef, batch_size=2, mode=mode
        )

    def make_symbolic_graph(self):
        x0 = self.make_and_register_variable("x0", shape=(3,))
        P0 = self.make_and_register_variable("P0", shape=(3, 3))

        ar_params = self.make_and_register_variable("ar_params", shape=(3,))
        sigma_x = self.make_and_register_variable("sigma_x", shape=(1,))

        self.ssm["transition", :, :] = np.eye(3, k=-1)
        self.ssm["selection", 0, 0] = 1
        self.ssm["design", 0, 0] = 1

        self.ssm["initial_state", :] = x0
        self.ssm["initial_state_cov", :, :] = P0
        self.ssm["transition", 0, :] = ar_params
        self.ssm["state_cov", :, :] = sigma_x

    def set_parameters(self):
        # Only the "name" parameter is required here. "Shape" is only used when printing the
        # model requirements table. "Dims" are used to link variables to coords.
        x0 = Parameter(name="x0", shape=(3,), dims=(ALL_STATE_DIM,))
        P0 = Parameter(name="P0", shape=(3, 3), dims=(ALL_STATE_DIM, ALL_STATE_AUX_DIM))

        ar_params = Parameter(
            name="ar_params", shape=(3,), dims=("ar_lags",), constraints="Stationary, please :)"
        )
        sigma_x = Parameter(name="sigma_x", shape=(1,), dims=(SHOCK_DIM,))
        return x0, P0, ar_params, sigma_x

    def set_states(self):
        # To get a name on the observed, we make an observed state
        ts1 = State(name="ts1", observed=True)

        # Since the three hidden states are lags of the data, i'll call them L1, L2 L3
        L1 = State(name="L1.data", observed=False)
        L2 = State(name="L2.data", observed=False)
        L3 = State(name="L3.data", observed=False)

        return ts1, L1, L2, L3

    def set_shocks(self):
        # There is one shock, called the "innovations" in the literature, so i'll go with that
        innovation = Shock(name="innovations")
        return innovation

    def set_coords(self):
        # This function sets up the coords dictionary used by pm.Model. The parent class has a helper
        # self.default_coords() that makes the coords that are always expected by a statespace model --
        # stuff like state, shock, etc.

        # You need to give one Coord object per dimension used among the Parameter objects you made a

        default_coords = self.default_coords()
        ar_coord = Coord(dimension="ar_lags", labels=(1, 2, 3))
        return *default_coords, ar_coord

In [ ]:
ar3 = AutoRegressiveThree(mode="NUMBA")

In [ ]:
data = np.random.normal(0, 1, size=(100, 2))

In [ ]:
batched_data = data.reshape(2, 100, 1)

In [ ]:
# Not vectorized
with pm.Model(coords=ar3.coords) as pymc_mod:
    x0 = pm.Deterministic("x0", pt.zeros((3,)), dims=("state"))
    P0 = pm.Deterministic("P0", pt.eye(3) * 10, dims=("state", "state_aux"))
    ar_params = pm.Normal("ar_params", shape=(3,), dims=("state"))

    sigma_x = pm.Exponential("sigma_x", 1, shape=(1,), dims=("shock"))

    ar3.build_statespace_graph(data=data[:, [1]])

In [ ]:
# Vectorized
with pm.Model(coords=ar3.coords | {"batch": ["batch_1", "batch_2"]}) as pymc_mod:
    x0 = pm.Deterministic("x0", pt.zeros((2, 3)), dims=("batch", "state"))
    P0 = pm.Deterministic(
        "P0", pt.tile(pt.eye(3) * 1, (2, 1, 1)), dims=("batch", "state", "state_aux")
    )
    ar_params = pm.Normal("ar_params", dims=("batch", "state"))

    sigma_x = pm.Exponential("sigma_x", 1, dims=("batch", "shock"))

    ar3.build_statespace_graph(data=batched_data)

In [ ]:
with pymc_mod:
    idata = pm.sample(tune=200, draws=200, compile_kwargs={"mode": "NUMBA"})

In [ ]:
post = ar3.sample_conditional_posterior(idata, mvn_method="cholesky")

In [ ]:
with pymc_mod:
    prior = pm.sample_prior_predictive(compile_kwargs={"mode": "NUMBA"})

In [ ]:
ar3.sample_conditional_prior(prior, mvn_method="cholesky")

In [ ]:
ar3.sample_unconditional_prior(prior, mvn_method="cholesky")

In [ ]:
unpost = ar3.sample_unconditional_posterior(idata, mvn_method="cholesky")

In [ ]:
unpost

# MVN Method

In [ ]:
class AutoRegressive3TwoSeries(PyMCStateSpace):
    def __init__(self, mode: str):
        k_states = 6  # 2 series × 3 lags
        k_posdef = 2  # one innovation per series
        k_endog = 2  # two observed series

        super().__init__(k_endog=k_endog, k_states=k_states, k_posdef=k_posdef, mode=mode)

    def make_symbolic_graph(self):
        x0 = self.make_and_register_variable("x0", shape=(6,))
        P0 = self.make_and_register_variable("P0", shape=(6, 6))

        ar_params = self.make_and_register_variable("ar_params", shape=(2, 3))
        sigma_x = self.make_and_register_variable("sigma_x", shape=(2,))

        T = np.eye(6, k=-1)

        self.ssm["transition", :, :] = T
        self.ssm["transition", 0, 0:3] = ar_params[0]
        self.ssm["transition", 3, 3:6] = ar_params[1]

        self.ssm["selection", 0, 0] = 1
        self.ssm["selection", 3, 1] = 1

        self.ssm["state_cov", :, :] = pt.diag(sigma_x)

        Z = np.zeros((2, 6))
        Z[0, 0] = 1
        Z[1, 3] = 1
        self.ssm["design", :, :] = Z

        self.ssm["initial_state", :] = x0
        self.ssm["initial_state_cov", :, :] = P0

    def set_parameters(self):
        x0 = Parameter(name="x0", shape=(6,), dims=(ALL_STATE_DIM,))
        P0 = Parameter(name="P0", shape=(6, 6), dims=(ALL_STATE_DIM, ALL_STATE_AUX_DIM))

        ar_params = Parameter(
            name="ar_params",
            shape=(2, 3),
            dims=("observed_state", "ar_lags"),
        )

        sigma_x = Parameter(
            name="sigma_x",
            shape=(2,),
            dims=("observed_state",),
        )

        return x0, P0, ar_params, sigma_x

    def set_states(self):
        # Observed states
        ts1 = State(name="ts1", observed=True)
        ts2 = State(name="ts2", observed=True)

        # Series 1 states
        L1_s1 = State(name="L1.ts1", observed=False)
        L2_s1 = State(name="L2.ts1", observed=False)
        L3_s1 = State(name="L3.ts1", observed=False)

        # Series 2 states
        L1_s2 = State(name="L1.ts2", observed=False)
        L2_s2 = State(name="L2.ts2", observed=False)
        L3_s2 = State(name="L3.ts2", observed=False)

        return (
            ts1,
            ts2,
            L1_s1,
            L2_s1,
            L3_s1,
            L1_s2,
            L2_s2,
            L3_s2,
        )

    def set_shocks(self):
        eps1 = Shock(name="innovation.ts1")
        eps2 = Shock(name="innovation.ts2")
        return eps1, eps2

    def set_coords(self):
        default_coords = self.default_coords()
        ar_coord = Coord(dimension="ar_lags", labels=(1, 2, 3))
        return *default_coords, ar_coord

In [ ]:
ar3.coords

In [ ]:
ar3.ssm["design"].eval()

In [ ]:
ar3 = AutoRegressive3TwoSeries(mode="NUMBA")

In [ ]:
with pm.Model(coords=ar3.coords) as pymc_mod:
    x0 = pm.Deterministic("x0", pt.zeros(6), dims=["state"])
    P0 = pm.Deterministic("P0", pt.eye(6) * 10, dims=["state", "state_aux"])

    # global mean per lag
    rho_global = pm.Normal("rho_global", 0.0, 0.5, dims=["ar_lags"])
    tau = pm.Exponential("tau", 2.0, dims=["ar_lags"])

    ar_offset = pm.Normal("ar_offset", 0.0, 1.0, dims=["observed_state", "ar_lags"])

    ar_params = pm.Deterministic(
        "ar_params",
        rho_global + tau * ar_offset,
        dims=["observed_state", "ar_lags"],
    )

    sigma_x = pm.Exponential("sigma_x", 1.0, dims=["observed_state"])

    ar3.build_statespace_graph(data=data)
    idata = pm.sample(compile_kwargs={"mode": "NUMBA"})

In [ ]:
pymc_mod.to_graphviz()

# MISC

In [ ]:
from pymc_extras.statespace import BayesianVARMAX

In [ ]:
def varma_mod(data):
    return BayesianVARMAX(
        endog_names=data.columns,
        order=(2, 0),
        stationary_initialization=True,
        verbose=False,
        measurement_error=True,
    )

In [ ]:
def idata(pymc_mod, rng):
    with pymc_mod:
        idata = pm.sample_prior_predictive(draws=10, random_seed=rng)

    return idata

In [ ]:
def pymc_mod(varma_mod, data):
    with pm.Model(coords=varma_mod.coords) as pymc_mod:
        state_chol, *_ = pm.LKJCholeskyCov(
            "state_chol", n=varma_mod.k_posdef, eta=1, sd_dist=pm.Exponential.dist(1)
        )
        ar_params = pm.Normal(
            "ar_params", mu=0, sigma=0.1, dims=["observed_state", "lag_ar", "observed_state_aux"]
        )
        state_cov = pm.Deterministic(
            "state_cov", state_chol @ state_chol.T, dims=["shock", "shock_aux"]
        )
        sigma_obs = pm.Exponential("sigma_obs", 1, dims=["observed_state"])

        varma_mod.build_statespace_graph(data=data, save_kalman_filter_outputs_in_idata=True)

    return pymc_mod

In [ ]:
rng = np.random.default_rng()

In [ ]:
import pandas as pd
import pytensor

floatX = pytensor.config.floatX

In [ ]:
def data():
    df = pd.read_csv(
        "../tests/statespace/_data/statsmodels_macrodata_processed.csv",
        index_col=0,
        parse_dates=True,
    ).astype(floatX)
    df.index.freq = df.index.inferred_freq
    return df

In [ ]:
data = data()

In [ ]:
varma_mod_ = varma_mod(data)
pymc_mod_ = pymc_mod(varma_mod_, data)
idata_ = idata(pymc_mod_, rng)

In [ ]:
def test_forecast(varma_mod, idata, rng):
    forecast = varma_mod.forecast(idata.prior, periods=10, random_seed=rng)

    assert np.isfinite(forecast.forecast_latent.values).all()
    assert np.isfinite(forecast.forecast_observed.values).all()